# EffoVPR_G - Kaggle Top 10 similar images

Attach Kaggle input datasets containing the EffoVPR artifact (`config.json` and `model.safetensors`) and your query images. Enable Internet in Kaggle Notebook settings so the notebook can download `PhamPhien/VN_Attractions`. EffoVPR source modules are embedded below. Global cosine similarity retrieves candidates; local MNN optionally reranks the Top 10.

Set `QUERY_DIR` in the configuration cell to the mounted folder containing your query images. The model artifact directory is discovered automatically under `/kaggle/input`.


## 1. Install runtime dependencies (run once per Kaggle session)

In [ ]:
%pip install -q "transformers==5.18.0" "datasets==5.0.1" "safetensors>=0.4.3" "pandas>=2.0" "Pillow>=10.0" "matplotlib>=3.7" "pyarrow>=15.0"

## 2. Load embedded EffoVPR code and detect Kaggle inputs

This cell writes source code only to `/kaggle/working`. It automatically finds the mounted EffoVPR model, `Index1/index/` gallery files, and images from `Image_Test_1`.


In [ ]:
from pathlib import Path
import os, sys, json, base64
CODE_ROOT = Path("/kaggle/working/effovpr_use_model").resolve()
CODE_ROOT.mkdir(parents=True, exist_ok=True)
MODULE_FILES_B64 = {'src/__init__.py': 'IiIiRWZmb1ZQUiBwYWNrYWdlLiIiIg0K', 'src/effovpr/__init__.py': 'X19hbGxfXyA9IFsiRWZmb1ZQUiIsICJDb3NGYWNlIiwgImJ1aWxkX2VmZm92cHJfbW9kZWwiXQ0KDQpmcm9tIC5tb2RlbHMuZWZmb3ZwciBpbXBvcnQgRWZmb1ZQUg0KZnJvbSAubW9kZWxzLmNvc2ZhY2UgaW1wb3J0IENvc0ZhY2UNCg0KDQpkZWYgYnVpbGRfZWZmb3Zwcl9tb2RlbCgqYXJncywgKiprd2FyZ3MpOg0KICAgIHJldHVybiBFZmZvVlBSKCphcmdzLCAqKmt3YXJncykNCg==', 'src/effovpr/models/cosface.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgdG9yY2gNCmltcG9ydCB0b3JjaC5ubiBhcyBubg0KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KDQoNCmNsYXNzIENvc0ZhY2Uobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fZmVhdHVyZXM6IGludCwgb3V0X2ZlYXR1cmVzOiBpbnQsIHNjYWxlOiBmbG9hdCA9IDMwLjAsIG1hcmdpbjogZmxvYXQgPSAwLjQpOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5pbl9mZWF0dXJlcyA9IGluX2ZlYXR1cmVzDQogICAgICAgIHNlbGYub3V0X2ZlYXR1cmVzID0gb3V0X2ZlYXR1cmVzDQogICAgICAgIHNlbGYuc2NhbGUgPSBzY2FsZQ0KICAgICAgICBzZWxmLm1hcmdpbiA9IG1hcmdpbg0KICAgICAgICBzZWxmLndlaWdodCA9IG5uLlBhcmFtZXRlcih0b3JjaC5lbXB0eShvdXRfZmVhdHVyZXMsIGluX2ZlYXR1cmVzKSkNCiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi53ZWlnaHQpDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCBlbWJlZGRpbmdzOiB0b3JjaC5UZW5zb3IsIHRhcmdldHM6IHRvcmNoLlRlbnNvcik6DQogICAgICAgIGlmIHRhcmdldHMgaXMgTm9uZToNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNvc0ZhY2UgcmVxdWlyZXMgdGFyZ2V0IGNsYXNzIGluZGljZXMiKQ0KICAgICAgICBpZiBlbWJlZGRpbmdzLmRpbSgpICE9IDI6DQogICAgICAgICAgICBlbWJlZGRpbmdzID0gZW1iZWRkaW5ncy52aWV3KGVtYmVkZGluZ3Muc2l6ZSgwKSwgLTEpDQogICAgICAgIGlmIGVtYmVkZGluZ3Muc2l6ZSgxKSAhPSBzZWxmLmluX2ZlYXR1cmVzOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkV4cGVjdGVkIGVtYmVkZGluZyBkaW1lbnNpb24ge3NlbGYuaW5fZmVhdHVyZXN9LCBnb3Qge2VtYmVkZGluZ3Muc2l6ZSgxKX0iKQ0KICAgICAgICBpZiB0YXJnZXRzLm5kaW0gIT0gMSBvciB0YXJnZXRzLnNpemUoMCkgIT0gZW1iZWRkaW5ncy5zaXplKDApOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidGFyZ2V0cyBtdXN0IGJlIGEgMUQgdGVuc29yIHdpdGggb25lIGNsYXNzIGluZGV4IHBlciBlbWJlZGRpbmciKQ0KICAgICAgICBpZiB0YXJnZXRzLmR0eXBlICE9IHRvcmNoLmxvbmc6DQogICAgICAgICAgICB0YXJnZXRzID0gdGFyZ2V0cy5sb25nKCkNCiAgICAgICAgaWYgdGFyZ2V0cy5udW1lbCgpIGFuZCAodGFyZ2V0cy5taW4oKSA8IDAgb3IgdGFyZ2V0cy5tYXgoKSA+PSBzZWxmLm91dF9mZWF0dXJlcyk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidGFyZ2V0cyBtdXN0IGJlIGluIFswLCB7c2VsZi5vdXRfZmVhdHVyZXN9KSIpDQogICAgICAgIG5vcm1fZW1iZWRkaW5ncyA9IEYubm9ybWFsaXplKGVtYmVkZGluZ3MsIHA9MiwgZGltPTEpDQogICAgICAgIG5vcm1fd2VpZ2h0ID0gRi5ub3JtYWxpemUoc2VsZi53ZWlnaHQsIHA9MiwgZGltPTEpDQogICAgICAgIGNvc2luZSA9IG5vcm1fZW1iZWRkaW5ncyBAIG5vcm1fd2VpZ2h0LlQNCiAgICAgICAgdGFyZ2V0X21hc2sgPSBGLm9uZV9ob3QodGFyZ2V0cywgbnVtX2NsYXNzZXM9c2VsZi5vdXRfZmVhdHVyZXMpLnRvKGR0eXBlPWNvc2luZS5kdHlwZSkNCiAgICAgICAgbG9naXRzID0gKGNvc2luZSAtIHNlbGYubWFyZ2luICogdGFyZ2V0X21hc2spICogc2VsZi5zY2FsZQ0KICAgICAgICBsb3NzID0gRi5jcm9zc19lbnRyb3B5KGxvZ2l0cywgdGFyZ2V0cykNCiAgICAgICAgcmV0dXJuIGxvZ2l0cywgbG9zcw0K', 'src/effovpr/models/dino_attention.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgbWF0aA0KDQppbXBvcnQgdG9yY2gNCg0KDQpkZWYgcmVzb2x2ZV9sYXllcl9pbmRleChsYXllciwgdG90YWxfbGF5ZXJzOiBpbnQpIC0+IGludDoNCiAgICBpZiBpc2luc3RhbmNlKGxheWVyLCBzdHIpOg0KICAgICAgICBsYXllciA9IGxheWVyLnN0cmlwKCkubG93ZXIoKQ0KICAgICAgICBpZiBsYXllci5zdGFydHN3aXRoKCJuLSIpOg0KICAgICAgICAgICAgb2Zmc2V0ID0gaW50KGxheWVyLnNwbGl0KCItIilbLTFdKQ0KICAgICAgICAgICAgaWR4ID0gdG90YWxfbGF5ZXJzIC0gb2Zmc2V0DQogICAgICAgIGVsaWYgbGF5ZXIgPT0gIm4iOg0KICAgICAgICAgICAgaWR4ID0gdG90YWxfbGF5ZXJzIC0gMQ0KICAgICAgICBlbGlmIGxheWVyLnN0YXJ0c3dpdGgoIm4iKSBhbmQgbGF5ZXJbMTpdLmlzZGlnaXQoKToNCiAgICAgICAgICAgIGlkeCA9IGludChsYXllclsxOl0pIC0gMQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuc3VwcG9ydGVkIGxheWVyIHRva2VuOiB7bGF5ZXJ9IikNCiAgICBlbHNlOg0KICAgICAgICBpZHggPSBpbnQobGF5ZXIpDQogICAgaWYgaWR4IDwgMDoNCiAgICAgICAgaWR4ID0gdG90YWxfbGF5ZXJzICsgaWR4DQogICAgaWYgaWR4IDwgMCBvciBpZHggPj0gdG90YWxfbGF5ZXJzOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTGF5ZXIgaW5kZXgge2lkeH0gaXMgb3V0IG9mIHJhbmdlIGZvciB0b3RhbCBsYXllcnMge3RvdGFsX2xheWVyc30uIikNCiAgICByZXR1cm4gaWR4DQoNCg0KZGVmIHBhcnNlX3Rva2VucyhoaWRkZW5fdG9rZW5zOiB0b3JjaC5UZW5zb3IpOg0KICAgIGlmIGhpZGRlbl90b2tlbnMubmRpbSAhPSAzOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgW2JhdGNoLCB0b2tlbnMsIGRpbV0sIGdvdCB7dHVwbGUoaGlkZGVuX3Rva2Vucy5zaGFwZSl9IikNCiAgICBiYXRjaF9zaXplID0gaGlkZGVuX3Rva2Vucy5zaXplKDApDQogICAgY2xzX3Rva2VuID0gaGlkZGVuX3Rva2Vuc1s6LCAwOjEsIDpdDQogICAgcmVnaXN0ZXJfdG9rZW5zID0gaGlkZGVuX3Rva2Vuc1s6LCAxOjUsIDpdIGlmIGhpZGRlbl90b2tlbnMuc2l6ZSgxKSA+IDUgZWxzZSBoaWRkZW5fdG9rZW5zWzosIDE6MSwgOl0NCiAgICBwYXRjaF90b2tlbnMgPSBoaWRkZW5fdG9rZW5zWzosIDU6LCA6XQ0KICAgIHJldHVybiBjbHNfdG9rZW4sIHJlZ2lzdGVyX3Rva2VucywgcGF0Y2hfdG9rZW5zDQoNCg0KZGVmIGF0dGVudGlvbl9wYXRjaF9zY29yZXMocV9wYXRjaDogdG9yY2guVGVuc29yLCBrX2NsczogdG9yY2guVGVuc29yLCB0ZW1wZXJhdHVyZTogZmxvYXQgfCBOb25lID0gTm9uZSk6DQogICAgaWYgcV9wYXRjaC5uZGltICE9IDI6DQogICAgICAgIHFfcGF0Y2ggPSBxX3BhdGNoLnJlc2hhcGUocV9wYXRjaC5zaXplKDApLCAtMSkNCiAgICBpZiB0ZW1wZXJhdHVyZSBpcyBOb25lOg0KICAgICAgICB0ZW1wZXJhdHVyZSA9IG1hdGguc3FydChxX3BhdGNoLnNpemUoLTEpKQ0KICAgIHNjb3JlcyA9IChxX3BhdGNoIEAga19jbHMucmVzaGFwZSgtMSkpIC8gdGVtcGVyYXR1cmUNCiAgICByZXR1cm4gdG9yY2guc29mdG1heChzY29yZXMsIGRpbT0tMSkNCg0KDQpkZWYgYXNzZXJ0X3BhdGNoX2NvdW50KHBhdGNoX3Rva2VuczogdG9yY2guVGVuc29yLCBpbWFnZV9yZXNvbHV0aW9uOiBpbnQgfCBOb25lID0gTm9uZSk6DQogICAgcGF0Y2hfY291bnQgPSBwYXRjaF90b2tlbnMuc2hhcGVbMV0NCiAgICBncmlkX2ggPSBpbnQocm91bmQobWF0aC5zcXJ0KHBhdGNoX2NvdW50KSkpDQogICAgZ3JpZF93ID0gcGF0Y2hfY291bnQgLy8gZ3JpZF9oIGlmIGdyaWRfaCA+IDAgZWxzZSAwDQogICAgaWYgaW1hZ2VfcmVzb2x1dGlvbiBpcyBub3QgTm9uZToNCiAgICAgICAgZXhwZWN0ZWQgPSAoaW1hZ2VfcmVzb2x1dGlvbiAvLyAxNCkgKiogMg0KICAgICAgICBpZiBwYXRjaF9jb3VudCAhPSBleHBlY3RlZDoNCiAgICAgICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKGYiRXhwZWN0ZWQgcGF0Y2ggY291bnQge2V4cGVjdGVkfSBmb3IgcmVzb2x1dGlvbiB7aW1hZ2VfcmVzb2x1dGlvbn0sIGdvdCB7cGF0Y2hfY291bnR9IikNCiAgICBlbHNlOg0KICAgICAgICBpZiBncmlkX2ggKiBncmlkX3cgIT0gcGF0Y2hfY291bnQ6DQogICAgICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcihmInBhdGNoIGNvdW50IG1pc21hdGNoOiB7cGF0Y2hfY291bnR9ICE9IHtncmlkX2h9ICoge2dyaWRfd30iKQ0K', 'src/effovpr/models/dino_backbone.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQoNCmltcG9ydCB0b3JjaA0KZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Db25maWcsIEF1dG9JbWFnZVByb2Nlc3NvciwgQXV0b01vZGVsCg0KDQpjbGFzcyBESU5PdjJCYWNrYm9uZSh0b3JjaC5ubi5Nb2R1bGUpOg0KICAgIGRlZiBfX2luaXRfXygNCiAgICAgICAgc2VsZiwNCiAgICAgICAgbW9kZWxfbmFtZTogc3RyID0gImZhY2Vib29rL2Rpbm92Mi13aXRoLXJlZ2lzdGVycy1sYXJnZSIsDQogICAgICAgIHdpdGhfcmVnaXN0ZXJzOiBib29sID0gVHJ1ZSwNCiAgICAgICAgY2FjaGVfZGlyOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBiYWNrYm9uZV9jb25maWc6IGRpY3QgfCBOb25lID0gTm9uZSwKICAgICAgICBwcm9jZXNzb3JfY29uZmlnOiBkaWN0IHwgTm9uZSA9IE5vbmUsCiAgICApOg0KICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkNCiAgICAgICAgc2VsZi5tb2RlbF9uYW1lID0gbW9kZWxfbmFtZQ0KICAgICAgICBzZWxmLndpdGhfcmVnaXN0ZXJzID0gd2l0aF9yZWdpc3RlcnMNCiAgICAgICAgaWYgYmFja2JvbmVfY29uZmlnIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBBdXRvTW9kZWwuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUsIGNhY2hlX2Rpcj1jYWNoZV9kaXIpCiAgICAgICAgICAgIHNlbGYuaW1hZ2VfcHJvY2Vzc29yID0gQXV0b0ltYWdlUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZChtb2RlbF9uYW1lLCBjYWNoZV9kaXI9Y2FjaGVfZGlyKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNvbmZpZyA9IGRpY3QoYmFja2JvbmVfY29uZmlnKQogICAgICAgICAgICBtb2RlbF90eXBlID0gY29uZmlnLnBvcCgibW9kZWxfdHlwZSIpCiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBBdXRvTW9kZWwuZnJvbV9jb25maWcoQXV0b0NvbmZpZy5mb3JfbW9kZWwobW9kZWxfdHlwZSwgKipjb25maWcpKQogICAgICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgaW1hZ2VfcHJvY2Vzc2luZ191dGlscwogICAgICAgICAgICBpbXBvcnQgdHJhbnNmb3JtZXJzCiAgICAgICAgICAgIHByb2Nlc3NvciA9IGRpY3QocHJvY2Vzc29yX2NvbmZpZykKICAgICAgICAgICAgcHJvY2Vzc29yX3R5cGUgPSBwcm9jZXNzb3IucG9wKCJpbWFnZV9wcm9jZXNzb3JfdHlwZSIpCiAgICAgICAgICAgIHByb2Nlc3Nvcl9jbGFzcyA9IGdldGF0dHIodHJhbnNmb3JtZXJzLCBwcm9jZXNzb3JfdHlwZSkKICAgICAgICAgICAgaWYgbm90IGlzc3ViY2xhc3MocHJvY2Vzc29yX2NsYXNzLCBpbWFnZV9wcm9jZXNzaW5nX3V0aWxzLkJhc2VJbWFnZVByb2Nlc3Nvcik6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5zdXBwb3J0ZWQgaW1hZ2UgcHJvY2Vzc29yOiB7cHJvY2Vzc29yX3R5cGV9IikKICAgICAgICAgICAgc2VsZi5pbWFnZV9wcm9jZXNzb3IgPSBwcm9jZXNzb3JfY2xhc3MuZnJvbV9kaWN0KHByb2Nlc3NvcikKICAgICAgICBzZWxmLmhpZGRlbl9kaW0gPSBzZWxmLm1vZGVsLmNvbmZpZy5oaWRkZW5fc2l6ZQ0KICAgICAgICBzZWxmLnBhdGNoX3NpemUgPSBzZWxmLm1vZGVsLmNvbmZpZy5wYXRjaF9zaXplDQogICAgICAgIHNlbGYubnVtX2xheWVycyA9IGxlbihzZWxmLmVuY29kZXJfbGF5ZXJzKCkpDQoNCiAgICBkZWYgZW5jb2Rlcl9sYXllcnMoc2VsZik6DQogICAgICAgIGVuY29kZXIgPSBnZXRhdHRyKHNlbGYubW9kZWwsICJlbmNvZGVyIiwgTm9uZSkNCiAgICAgICAgaWYgZW5jb2RlciBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIGxheWVycyA9IGdldGF0dHIoZW5jb2RlciwgImxheWVyIiwgZ2V0YXR0cihlbmNvZGVyLCAibGF5ZXJzIiwgTm9uZSkpDQogICAgICAgICAgICBpZiBsYXllcnMgaXMgbm90IE5vbmU6DQogICAgICAgICAgICAgICAgcmV0dXJuIGxheWVycw0KICAgICAgICB2aXQgPSBnZXRhdHRyKHNlbGYubW9kZWwsICJ2aXQiLCBOb25lKQ0KICAgICAgICBlbmNvZGVyID0gZ2V0YXR0cih2aXQsICJlbmNvZGVyIiwgTm9uZSkNCiAgICAgICAgaWYgZW5jb2RlciBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIGxheWVycyA9IGdldGF0dHIoZW5jb2RlciwgImxheWVycyIsIGdldGF0dHIoZW5jb2RlciwgImxheWVyIiwgTm9uZSkpDQogICAgICAgICAgICBpZiBsYXllcnMgaXMgbm90IE5vbmU6DQogICAgICAgICAgICAgICAgcmV0dXJuIGxheWVycw0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJVbnN1cHBvcnRlZCBESU5PdjIgZW5jb2RlciBzdHJ1Y3R1cmUgZm9yIHt0eXBlKHNlbGYubW9kZWwpLl9fbmFtZV9ffSIpDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCBwaXhlbF92YWx1ZXM6IHRvcmNoLlRlbnNvciwgKiprd2FyZ3MpOg0KICAgICAgICByZXR1cm4gc2VsZi5tb2RlbChwaXhlbF92YWx1ZXM9cGl4ZWxfdmFsdWVzLCBvdXRwdXRfaGlkZGVuX3N0YXRlcz1UcnVlLCByZXR1cm5fZGljdD1UcnVlLCAqKmt3YXJncykNCg0KICAgIGRlZiBjb25maWd1cmVfdHJhaW5hYmxlX2xheWVycyhzZWxmLCBsYXN0X25fbGF5ZXJzOiBpbnQgPSA1KToNCiAgICAgICAgaWYgbGFzdF9uX2xheWVycyA8IDAgb3IgbGFzdF9uX2xheWVycyA+IHNlbGYubnVtX2xheWVyczoNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJsYXN0X25fbGF5ZXJzIG11c3QgYmUgYmV0d2VlbiAwIGFuZCB7c2VsZi5udW1fbGF5ZXJzfSIpDQogICAgICAgIGZvciBwYXJhbSBpbiBzZWxmLm1vZGVsLnBhcmFtZXRlcnMoKToNCiAgICAgICAgICAgIHBhcmFtLnJlcXVpcmVzX2dyYWQgPSBGYWxzZQ0KICAgICAgICBsYXllcnMgPSBzZWxmLmVuY29kZXJfbGF5ZXJzKCkNCiAgICAgICAgZm9yIGxheWVyIGluIGxpc3QobGF5ZXJzKVstbGFzdF9uX2xheWVyczpdIGlmIGxhc3Rfbl9sYXllcnMgZWxzZSBbXToNCiAgICAgICAgICAgIGZvciBwYXJhbSBpbiBsYXllci5wYXJhbWV0ZXJzKCk6DQogICAgICAgICAgICAgICAgcGFyYW0ucmVxdWlyZXNfZ3JhZCA9IFRydWUNCg==', 'src/effovpr/models/projection.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgdG9yY2gNCmltcG9ydCB0b3JjaC5ubiBhcyBubg0KDQoNCmNsYXNzIEdsb2JhbFByb2plY3Rpb24obm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRfZGltOiBpbnQgPSAxMDI0LCBvdXRwdXRfZGltOiBpbnQgPSAxMDI0KToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIGlmIG91dHB1dF9kaW0gbm90IGluICgxMjgsIDI1NiwgMTAyNCk6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTdXBwb3J0ZWQgZ2xvYmFsIGRpbWVuc2lvbnMgYXJlIDEyOCwgMjU2LCBhbmQgMTAyNC4iKQ0KICAgICAgICBzZWxmLmlucHV0X2RpbSA9IGlucHV0X2RpbQ0KICAgICAgICBzZWxmLm91dHB1dF9kaW0gPSBvdXRwdXRfZGltDQogICAgICAgIHNlbGYucHJvaiA9IG5uLkxpbmVhcihpbnB1dF9kaW0sIG91dHB1dF9kaW0pDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoNCiAgICAgICAgcmV0dXJuIHNlbGYucHJvaih4KQ0K', 'src/effovpr/models/effovpr.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQpmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgT3B0aW9uYWwNCg0KaW1wb3J0IHRvcmNoDQppbXBvcnQgdG9yY2gubm4gYXMgbm4NCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYNCg0KZnJvbSAuY29zZmFjZSBpbXBvcnQgQ29zRmFjZQ0KZnJvbSAuZGlub19hdHRlbnRpb24gaW1wb3J0IGFzc2VydF9wYXRjaF9jb3VudCwgYXR0ZW50aW9uX3BhdGNoX3Njb3JlcywgcGFyc2VfdG9rZW5zLCByZXNvbHZlX2xheWVyX2luZGV4DQpmcm9tIC5kaW5vX2JhY2tib25lIGltcG9ydCBESU5PdjJCYWNrYm9uZQ0KZnJvbSAucHJvamVjdGlvbiBpbXBvcnQgR2xvYmFsUHJvamVjdGlvbg0KDQoNCmNsYXNzIEVmZm9WUFIobm4uTW9kdWxlKToNCiAgICBkZWYgX19pbml0X18oDQogICAgICAgIHNlbGYsDQogICAgICAgIGJhY2tib25lX25hbWU6IHN0ciA9ICJmYWNlYm9vay9kaW5vdjItd2l0aC1yZWdpc3RlcnMtbGFyZ2UiLA0KICAgICAgICBnbG9iYWxfZGltOiBpbnQgPSAxMDI0LA0KICAgICAgICBudW1fY2xhc3NlczogT3B0aW9uYWxbaW50XSA9IE5vbmUsDQogICAgICAgIGZpbmVfdHVuZV9sYXN0X25fbGF5ZXJzOiBpbnQgPSA1LA0KICAgICAgICB3aXRoX3JlZ2lzdGVyczogYm9vbCA9IFRydWUsDQogICAgICAgIGNvc2ZhY2Vfc2NhbGU6IGZsb2F0ID0gMzAuMCwNCiAgICAgICAgY29zZmFjZV9tYXJnaW46IGZsb2F0ID0gMC40LA0KICAgICAgICBjYWNoZV9kaXI6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIGJhY2tib25lX2NvbmZpZzogZGljdCB8IE5vbmUgPSBOb25lLAogICAgICAgIHByb2Nlc3Nvcl9jb25maWc6IGRpY3QgfCBOb25lID0gTm9uZSwKICAgICk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBzZWxmLmJhY2tib25lID0gRElOT3YyQmFja2JvbmUobW9kZWxfbmFtZT1iYWNrYm9uZV9uYW1lLCB3aXRoX3JlZ2lzdGVycz13aXRoX3JlZ2lzdGVycywgY2FjaGVfZGlyPWNhY2hlX2RpciwgYmFja2JvbmVfY29uZmlnPWJhY2tib25lX2NvbmZpZywgcHJvY2Vzc29yX2NvbmZpZz1wcm9jZXNzb3JfY29uZmlnKQogICAgICAgIHNlbGYuZ2xvYmFsX2RpbSA9IGdsb2JhbF9kaW0NCiAgICAgICAgc2VsZi5nbG9iYWxfcHJvamVjdGlvbiA9IEdsb2JhbFByb2plY3Rpb24oaW5wdXRfZGltPXNlbGYuYmFja2JvbmUuaGlkZGVuX2RpbSwgb3V0cHV0X2RpbT1nbG9iYWxfZGltKQ0KICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gbnVtX2NsYXNzZXMNCiAgICAgICAgc2VsZi5jb3NmYWNlID0gQ29zRmFjZShpbl9mZWF0dXJlcz1nbG9iYWxfZGltLCBvdXRfZmVhdHVyZXM9bnVtX2NsYXNzZXMsIHNjYWxlPWNvc2ZhY2Vfc2NhbGUsIG1hcmdpbj1jb3NmYWNlX21hcmdpbikgaWYgbnVtX2NsYXNzZXMgaXMgbm90IE5vbmUgZWxzZSBOb25lDQogICAgICAgIHNlbGYuY29uZmlndXJlX3RyYWluYWJsZV9sYXllcnMobGFzdF9uX2xheWVycz1maW5lX3R1bmVfbGFzdF9uX2xheWVycykNCg0KICAgIGRlZiBjb25maWd1cmVfdHJhaW5hYmxlX2xheWVycyhzZWxmLCBsYXN0X25fbGF5ZXJzOiBpbnQgPSA1KToKICAgICAgICBzZWxmLmZpbmVfdHVuZV9sYXN0X25fbGF5ZXJzID0gbGFzdF9uX2xheWVycwogICAgICAgIGlmIGxhc3Rfbl9sYXllcnMgPCAwIG9yIGxhc3Rfbl9sYXllcnMgPiBzZWxmLmJhY2tib25lLm51bV9sYXllcnM6DQogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYibGFzdF9uX2xheWVycyBtdXN0IGJlIGJldHdlZW4gMCBhbmQge3NlbGYuYmFja2JvbmUubnVtX2xheWVyc30iKQ0KICAgICAgICBzZWxmLmJhY2tib25lLmNvbmZpZ3VyZV90cmFpbmFibGVfbGF5ZXJzKGxhc3Rfbl9sYXllcnM9bGFzdF9uX2xheWVycykNCiAgICAgICAgZm9yIHBhcmFtIGluIHNlbGYuZ2xvYmFsX3Byb2plY3Rpb24ucGFyYW1ldGVycygpOg0KICAgICAgICAgICAgcGFyYW0ucmVxdWlyZXNfZ3JhZCA9IFRydWUNCiAgICAgICAgaWYgc2VsZi5jb3NmYWNlIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgZm9yIHBhcmFtIGluIHNlbGYuY29zZmFjZS5wYXJhbWV0ZXJzKCk6DQogICAgICAgICAgICAgICAgcGFyYW0ucmVxdWlyZXNfZ3JhZCA9IFRydWUNCg0KICAgIGRlZiBfcHJlcGFyZV9pbWFnZXMoc2VsZiwgaW1hZ2VzKToNCiAgICAgICAgaWYgaXNpbnN0YW5jZShpbWFnZXMsIHRvcmNoLlRlbnNvcik6DQogICAgICAgICAgICByZXR1cm4gaW1hZ2VzDQogICAgICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2VzLCBsaXN0KSBhbmQgbGVuKGltYWdlcykgPiAwIGFuZCBpc2luc3RhbmNlKGltYWdlc1swXSwgdG9yY2guVGVuc29yKToNCiAgICAgICAgICAgIHJldHVybiB0b3JjaC5zdGFjayhpbWFnZXMsIGRpbT0wKQ0KICAgICAgICBwcm9jZXNzZWQgPSBzZWxmLmJhY2tib25lLmltYWdlX3Byb2Nlc3NvcihpbWFnZXM9aW1hZ2VzLCByZXR1cm5fdGVuc29ycz0icHQiKQ0KICAgICAgICByZXR1cm4gcHJvY2Vzc2VkWyJwaXhlbF92YWx1ZXMiXQ0KDQogICAgZGVmIF9hcHBseV9wcm9qZWN0aW9uKHNlbGYsIGNsc190b2tlbjogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6DQogICAgICAgIGZlYXQgPSBzZWxmLmdsb2JhbF9wcm9qZWN0aW9uKGNsc190b2tlbikNCiAgICAgICAgcmV0dXJuIEYubm9ybWFsaXplKGZlYXQsIHA9MiwgZGltPS0xKQ0KDQogICAgZGVmIGV4dHJhY3RfZ2xvYmFsKHNlbGYsIGltYWdlKToNCiAgICAgICAgcGl4ZWxfdmFsdWVzID0gc2VsZi5fcHJlcGFyZV9pbWFnZXMoaW1hZ2UpDQogICAgICAgIHBpeGVsX3ZhbHVlcyA9IHBpeGVsX3ZhbHVlcy50byhuZXh0KHNlbGYucGFyYW1ldGVycygpKS5kZXZpY2UpDQogICAgICAgIG91dHB1dHMgPSBzZWxmLmJhY2tib25lKHBpeGVsX3ZhbHVlcykNCiAgICAgICAgaGlkZGVuID0gb3V0cHV0cy5sYXN0X2hpZGRlbl9zdGF0ZQ0KICAgICAgICBjbHNfdG9rZW4gPSBoaWRkZW5bOiwgMCwgOl0NCiAgICAgICAgcmV0dXJuIHNlbGYuX2FwcGx5X3Byb2plY3Rpb24oY2xzX3Rva2VuKQ0KDQogICAgZGVmIGV4dHJhY3RfZ2xvYmFsX2FuZF9sb2NhbChzZWxmLCBpbWFnZSwgbGF5ZXI6IGludCB8IHN0ciA9ICJuLTEiKToNCiAgICAgICAgcGl4ZWxfdmFsdWVzID0gc2VsZi5fcHJlcGFyZV9pbWFnZXMoaW1hZ2UpDQogICAgICAgIHBpeGVsX3ZhbHVlcyA9IHBpeGVsX3ZhbHVlcy50byhuZXh0KHNlbGYucGFyYW1ldGVycygpKS5kZXZpY2UpDQogICAgICAgIG91dHB1dHMgPSBzZWxmLmJhY2tib25lKHBpeGVsX3ZhbHVlcykNCiAgICAgICAgaGlkZGVuX3N0YXRlcyA9IG91dHB1dHMuaGlkZGVuX3N0YXRlcw0KICAgICAgICBsYXllcl9pbmRleCA9IHJlc29sdmVfbGF5ZXJfaW5kZXgobGF5ZXIsIHNlbGYuYmFja2JvbmUubnVtX2xheWVycykNCiAgICAgICAgc2VsZWN0ZWQgPSBoaWRkZW5fc3RhdGVzW2xheWVyX2luZGV4ICsgMV0NCiAgICAgICAgY2xzX3Rva2VuLCByZWdpc3Rlcl90b2tlbnMsIHBhdGNoX3Rva2VucyA9IHBhcnNlX3Rva2VucyhzZWxlY3RlZCkNCiAgICAgICAgcGF0Y2hfdG9rZW5zID0gcGF0Y2hfdG9rZW5zLnJlc2hhcGUocGl4ZWxfdmFsdWVzLnNpemUoMCksIC0xLCBwYXRjaF90b2tlbnMuc2l6ZSgtMSkpDQogICAgICAgIGFzc2VydF9wYXRjaF9jb3VudChwYXRjaF90b2tlbnMsIGltYWdlX3Jlc29sdXRpb249aW50KHBpeGVsX3ZhbHVlcy5zaGFwZVstMV0pIGlmIHBpeGVsX3ZhbHVlcy5zaGFwZVstMV0gJSBzZWxmLmJhY2tib25lLnBhdGNoX3NpemUgPT0gMCBlbHNlIE5vbmUpDQogICAgICAgIGdsb2JhbF9mZWF0ID0gc2VsZi5fYXBwbHlfcHJvamVjdGlvbihvdXRwdXRzLmxhc3RfaGlkZGVuX3N0YXRlWzosIDAsIDpdKQ0KICAgICAgICByZXR1cm4gZ2xvYmFsX2ZlYXQsIHsibGF5ZXIiOiBsYXllcl9pbmRleCwgImNsc190b2tlbiI6IGNsc190b2tlbiwgInJlZ2lzdGVyX3Rva2VucyI6IHJlZ2lzdGVyX3Rva2VucywgInBhdGNoX3Rva2VucyI6IHBhdGNoX3Rva2Vuc30NCg0KICAgIGRlZiBleHRyYWN0X3FrdihzZWxmLCBpbWFnZSwgbGF5ZXI6IGludCB8IHN0ciA9ICJuLTEiKToNCiAgICAgICAgcGl4ZWxfdmFsdWVzID0gc2VsZi5fcHJlcGFyZV9pbWFnZXMoaW1hZ2UpDQogICAgICAgIHBpeGVsX3ZhbHVlcyA9IHBpeGVsX3ZhbHVlcy50byhuZXh0KHNlbGYucGFyYW1ldGVycygpKS5kZXZpY2UpDQogICAgICAgIG91dHB1dHMgPSBzZWxmLmJhY2tib25lKHBpeGVsX3ZhbHVlcykNCiAgICAgICAgaGlkZGVuX3N0YXRlcyA9IG91dHB1dHMuaGlkZGVuX3N0YXRlcw0KICAgICAgICBsYXllcl9pbmRleCA9IHJlc29sdmVfbGF5ZXJfaW5kZXgobGF5ZXIsIHNlbGYuYmFja2JvbmUubnVtX2xheWVycykNCiAgICAgICAgaW5wdXRfdG9rZW5zID0gaGlkZGVuX3N0YXRlc1tsYXllcl9pbmRleF0NCiAgICAgICAgYmxvY2sgPSBzZWxmLmJhY2tib25lLmVuY29kZXJfbGF5ZXJzKClbbGF5ZXJfaW5kZXhdDQogICAgICAgIGlmIGhhc2F0dHIoYmxvY2ssICJub3JtMSIpOg0KICAgICAgICAgICAgYXR0ZW50aW9uX2lucHV0ID0gYmxvY2subm9ybTEoaW5wdXRfdG9rZW5zKQ0KICAgICAgICBlbGlmIGhhc2F0dHIoYmxvY2ssICJsYXllcm5vcm1fYmVmb3JlIik6DQogICAgICAgICAgICBhdHRlbnRpb25faW5wdXQgPSBibG9jay5sYXllcm5vcm1fYmVmb3JlKGlucHV0X3Rva2VucykNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkNhbm5vdCBsb2NhdGUgcHJlLWF0dGVudGlvbiBub3JtYWxpemF0aW9uIGluIHt0eXBlKGJsb2NrKS5fX25hbWVfX30iKQ0KICAgICAgICBhdHRlbnRpb24gPSBnZXRhdHRyKGJsb2NrLCAiYXR0ZW50aW9uIiwgZ2V0YXR0cihibG9jaywgInNlbGZfYXR0biIsIE5vbmUpKQ0KICAgICAgICBpZiBhdHRlbnRpb24gaXMgTm9uZToNCiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkNhbm5vdCBsb2NhdGUgYXR0ZW50aW9uIG1vZHVsZSBpbiB7dHlwZShibG9jaykuX19uYW1lX199IikNCiAgICAgICAgIyBIdWdnaW5nIEZhY2UncyBESU5PdjIgaW1wbGVtZW50YXRpb25zIGV4cG9zZSBlaXRoZXIgcV9wcm9qL2tfcHJvai92X3Byb2oNCiAgICAgICAgIyBkaXJlY3RseSBvciBxdWVyeS9rZXkvdmFsdWUgdW5kZXIgYW4gaW5uZXIgYXR0ZW50aW9uIG1vZHVsZS4NCiAgICAgICAgcHJvamVjdGlvbnMgPSBbZ2V0YXR0cihhdHRlbnRpb24sIG5hbWUsIE5vbmUpIGZvciBuYW1lIGluICgicV9wcm9qIiwgImtfcHJvaiIsICJ2X3Byb2oiKV0NCiAgICAgICAgaWYgYWxsKHByb2plY3Rpb24gaXMgbm90IE5vbmUgZm9yIHByb2plY3Rpb24gaW4gcHJvamVjdGlvbnMpOg0KICAgICAgICAgICAgcV9wcm9qLCBrX3Byb2osIHZfcHJvaiA9IHByb2plY3Rpb25zDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBpbm5lcl9hdHRlbnRpb24gPSBnZXRhdHRyKGF0dGVudGlvbiwgImF0dGVudGlvbiIsIGF0dGVudGlvbikNCiAgICAgICAgICAgIHByb2plY3Rpb25zID0gW2dldGF0dHIoaW5uZXJfYXR0ZW50aW9uLCBuYW1lLCBOb25lKSBmb3IgbmFtZSBpbiAoInF1ZXJ5IiwgImtleSIsICJ2YWx1ZSIpXQ0KICAgICAgICAgICAgaWYgbm90IGFsbChwcm9qZWN0aW9uIGlzIG5vdCBOb25lIGZvciBwcm9qZWN0aW9uIGluIHByb2plY3Rpb25zKToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJVbnN1cHBvcnRlZCBRL0svViBwcm9qZWN0aW9uIHN0cnVjdHVyZSBpbiB7dHlwZShhdHRlbnRpb24pLl9fbmFtZV9ffSIpDQogICAgICAgICAgICBxX3Byb2osIGtfcHJvaiwgdl9wcm9qID0gcHJvamVjdGlvbnMNCiAgICAgICAgcSA9IHFfcHJvaihhdHRlbnRpb25faW5wdXQpDQogICAgICAgIGsgPSBrX3Byb2ooYXR0ZW50aW9uX2lucHV0KQ0KICAgICAgICB2ID0gdl9wcm9qKGF0dGVudGlvbl9pbnB1dCkNCiAgICAgICAgY2xzX3Rva2VuLCByZWdpc3Rlcl90b2tlbnMsIF8gPSBwYXJzZV90b2tlbnMocSkNCiAgICAgICAgdl9wYXRjaF90b2tlbnMgPSBwYXJzZV90b2tlbnModilbMl0NCiAgICAgICAgYXNzZXJ0X3BhdGNoX2NvdW50KHZfcGF0Y2hfdG9rZW5zLCBpbWFnZV9yZXNvbHV0aW9uPWludChwaXhlbF92YWx1ZXMuc2hhcGVbLTFdKSBpZiBwaXhlbF92YWx1ZXMuc2hhcGVbLTFdICUgc2VsZi5iYWNrYm9uZS5wYXRjaF9zaXplID09IDAgZWxzZSBOb25lKQ0KICAgICAgICByZXR1cm4gew0KICAgICAgICAgICAgInEiOiBxLA0KICAgICAgICAgICAgImsiOiBrLA0KICAgICAgICAgICAgInYiOiB2LA0KICAgICAgICAgICAgImNsc190b2tlbiI6IGNsc190b2tlbiwNCiAgICAgICAgICAgICJyZWdpc3Rlcl90b2tlbnMiOiByZWdpc3Rlcl90b2tlbnMsDQogICAgICAgICAgICAicV9wYXRjaCI6IHBhcnNlX3Rva2VucyhxKVsyXSwNCiAgICAgICAgICAgICJrX3BhdGNoIjogcGFyc2VfdG9rZW5zKGspWzJdLA0KICAgICAgICAgICAgInZfcGF0Y2giOiBwYXJzZV90b2tlbnModilbMl0sDQogICAgICAgICAgICAibGF5ZXIiOiBsYXllcl9pbmRleCwNCiAgICAgICAgfQ0KDQogICAgZGVmIGNvbXB1dGVfYXR0ZW50aW9uX3Njb3JlKHNlbGYsIHFrdjogRGljdFtzdHIsIHRvcmNoLlRlbnNvcl0sIFQxOiBmbG9hdCA9IDAuMDUpOg0KICAgICAgICBxX3BhdGNoID0gcWt2WyJxX3BhdGNoIl1bMF0NCiAgICAgICAga19jbHMgPSBxa3ZbImsiXVswLCAwLCA6XQ0KICAgICAgICBzY29yZXMgPSBhdHRlbnRpb25fcGF0Y2hfc2NvcmVzKHFfcGF0Y2gsIGtfY2xzKQ0KICAgICAgICBzZWxlY3RlZCA9IHNjb3JlcyA+IFQxDQogICAgICAgIHNlbGVjdGVkX3YgPSBxa3ZbInZfcGF0Y2giXVswXVtzZWxlY3RlZF0NCiAgICAgICAgcmV0dXJuIHNjb3Jlcywgc2VsZWN0ZWQsIHNlbGVjdGVkX3YNCg0KICAgIGRlZiBmb3J3YXJkKHNlbGYsIGltYWdlcywgbGFiZWxzPU5vbmUsICoqa3dhcmdzKToNCiAgICAgICAgcGl4ZWxfdmFsdWVzID0gc2VsZi5fcHJlcGFyZV9pbWFnZXMoaW1hZ2VzKQ0KICAgICAgICBvdXRwdXRzID0gc2VsZi5iYWNrYm9uZShwaXhlbF92YWx1ZXMpDQogICAgICAgIGhpZGRlbiA9IG91dHB1dHMubGFzdF9oaWRkZW5fc3RhdGUNCiAgICAgICAgY2xzX3Rva2VuID0gaGlkZGVuWzosIDAsIDpdDQogICAgICAgIGdsb2JhbF9mZWF0ID0gc2VsZi5fYXBwbHlfcHJvamVjdGlvbihjbHNfdG9rZW4pDQogICAgICAgIGlmIGxhYmVscyBpcyBub3QgTm9uZSBhbmQgc2VsZi5jb3NmYWNlIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgbG9naXRzLCBsb3NzID0gc2VsZi5jb3NmYWNlKGdsb2JhbF9mZWF0LCBsYWJlbHMpDQogICAgICAgICAgICByZXR1cm4gZ2xvYmFsX2ZlYXQsIGxvZ2l0cywgbG9zcw0KICAgICAgICByZXR1cm4gZ2xvYmFsX2ZlYXQNCg==', 'src/effovpr/utils/model_artifact.py': 'IiIiQ29tcGxldGUsIG9mZmxpbmUgRWZmb1ZQUiBkaXN0cmlidXRpb24gYXJ0aWZhY3RzOyBjaGVja3BvaW50cyByZW1haW4gdHJhaW5pbmctb25seS4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRyYW5zZm9ybWVycwpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKZnJvbSBzYWZldGVuc29ycy50b3JjaCBpbXBvcnQgbG9hZF9maWxlLCBzYXZlX2ZpbGUKCmZyb20gLi5tb2RlbHMuZWZmb3ZwciBpbXBvcnQgRWZmb1ZQUgpmcm9tIC4ubW9kZWxzLmRpbm9fYXR0ZW50aW9uIGltcG9ydCBhdHRlbnRpb25fcGF0Y2hfc2NvcmVzCmZyb20gLi5yZXJhbmtpbmcubW5uIGltcG9ydCBtdXR1YWxfbmVhcmVzdF9uZWlnaGJvcl9tYXRjaGVzCgoKZGVmIGxvYWRfbW9kZWxfYXJ0aWZhY3QobW9kZWxfZGlyLCBkZXZpY2U9ImNwdSIpOgogICAgbW9kZWxfZGlyID0gUGF0aChtb2RlbF9kaXIpCiAgICBjb25maWcgPSBqc29uLmxvYWRzKChtb2RlbF9kaXIgLyAiY29uZmlnLmpzb24iKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICBpZiBjb25maWdbImZvcm1hdF92ZXJzaW9uIl0gIT0gMSBvciBjb25maWdbImFyY2hpdGVjdHVyZSJdICE9ICJFZmZvVlBSIjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVbnN1cHBvcnRlZCBFZmZvVlBSIGFydGlmYWN0IGZvcm1hdCIpCiAgICBtb2RlbCA9IEVmZm9WUFIoKipjb25maWdbIm1vZGVsIl0sIGJhY2tib25lX2NvbmZpZz1jb25maWdbImJhY2tib25lX2NvbmZpZyJdLAogICAgICAgICAgICAgICAgICAgIHByb2Nlc3Nvcl9jb25maWc9Y29uZmlnWyJwcm9jZXNzb3JfY29uZmlnIl0pCiAgICBpZiBtb2RlbC5iYWNrYm9uZS5oaWRkZW5fZGltICE9IGNvbmZpZ1siaGlkZGVuX2RpbSJdIG9yIG1vZGVsLmJhY2tib25lLnBhdGNoX3NpemUgIT0gY29uZmlnWyJwYXRjaF9zaXplIl06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQXJ0aWZhY3QgYXJjaGl0ZWN0dXJlIG1ldGFkYXRhIG1pc21hdGNoIikKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChsb2FkX2ZpbGUoc3RyKG1vZGVsX2RpciAvICJtb2RlbC5zYWZldGVuc29ycyIpKSwgc3RyaWN0PVRydWUpCiAgICByZXR1cm4gbW9kZWwudG8oZGV2aWNlKS5ldmFsKCksIGNvbmZpZwoKCmRlZiBwcmVwYXJlX2ltYWdlKG1vZGVsLCBpbWFnZSwgcmVzb2x1dGlvbik6CiAgICBpZiBpc2luc3RhbmNlKGltYWdlLCAoc3RyLCBQYXRoKSk6CiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKGltYWdlKSBhcyBvcGVuZWQ6CiAgICAgICAgICAgIGltYWdlID0gb3BlbmVkLmNvbnZlcnQoIlJHQiIpCiAgICBpZiByZXNvbHV0aW9uIDw9IDAgb3IgcmVzb2x1dGlvbiAlIG1vZGVsLmJhY2tib25lLnBhdGNoX3NpemU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVzb2x1dGlvbiBtdXN0IGJlIGEgcG9zaXRpdmUgbXVsdGlwbGUgb2YgcGF0Y2ggc2l6ZSIpCiAgICBpbWFnZSA9IGltYWdlLmNvbnZlcnQoIlJHQiIpLnJlc2l6ZSgocmVzb2x1dGlvbiwgcmVzb2x1dGlvbiksIEltYWdlLlJlc2FtcGxpbmcuQklDVUJJQykKICAgIHJldHVybiBtb2RlbC5iYWNrYm9uZS5pbWFnZV9wcm9jZXNzb3IoaW1hZ2VzPVtpbWFnZV0sIGRvX3Jlc2l6ZT1GYWxzZSwKICAgICAgICBkb19jZW50ZXJfY3JvcD1GYWxzZSwgcmV0dXJuX3RlbnNvcnM9InB0IilbInBpeGVsX3ZhbHVlcyJdLnRvKG5leHQobW9kZWwucGFyYW1ldGVycygpKS5kZXZpY2UpCgoKQHRvcmNoLmluZmVyZW5jZV9tb2RlKCkKZGVmIHZlcmlmeV9tb2RlbF9hcnRpZmFjdChvcmlnaW5hbCwgbW9kZWxfZGlyLCBpbWFnZSk6CiAgICByZXN0b3JlZCwgY29uZmlnID0gbG9hZF9tb2RlbF9hcnRpZmFjdChtb2RlbF9kaXIsIG5leHQob3JpZ2luYWwucGFyYW1ldGVycygpKS5kZXZpY2UpCiAgICBvcmlnaW5hbC5ldmFsKCkKICAgIHBpeGVscyA9IHByZXBhcmVfaW1hZ2Uob3JpZ2luYWwsIGltYWdlLCBjb25maWdbImluZmVyZW5jZSJdWyJyZXNvbHV0aW9uIl0pCiAgICByZXN0b3JlZF9waXhlbHMgPSBwcmVwYXJlX2ltYWdlKHJlc3RvcmVkLCBpbWFnZSwgY29uZmlnWyJpbmZlcmVuY2UiXVsicmVzb2x1dGlvbiJdKQogICAgdG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UocGl4ZWxzLCByZXN0b3JlZF9waXhlbHMsIHJ0b2w9MCwgYXRvbD0wKQogICAgZmlyc3QgPSBvcmlnaW5hbC5leHRyYWN0X2dsb2JhbChwaXhlbHMpCiAgICBzZWNvbmQgPSByZXN0b3JlZC5leHRyYWN0X2dsb2JhbChyZXN0b3JlZF9waXhlbHMpCiAgICBhc3NlcnQgZmlyc3Quc2hhcGUgPT0gc2Vjb25kLnNoYXBlID09ICgxLCBjb25maWdbIm1vZGVsIl1bImdsb2JhbF9kaW0iXSkKICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGZpcnN0LCBzZWNvbmQsIHJ0b2w9MWUtNSwgYXRvbD0xZS02KQogICAgcmVwb3J0ID0geyJnbG9iYWxfc2hhcGUiOiBsaXN0KHNlY29uZC5zaGFwZSksICJtYXhfYWJzb2x1dGVfZGlmZmVyZW5jZSI6IGZsb2F0KChmaXJzdC1zZWNvbmQpLmFicygpLm1heCgpKSwKICAgICAgICAgICAgICAiY29zaW5lX3NpbWlsYXJpdHkiOiBmbG9hdCh0b3JjaC5ubi5mdW5jdGlvbmFsLmNvc2luZV9zaW1pbGFyaXR5KGZpcnN0LCBzZWNvbmQpLml0ZW0oKSl9CiAgICBzZXR0aW5ncyA9IGNvbmZpZ1sicmVyYW5raW5nIl0KICAgIGEgPSBvcmlnaW5hbC5leHRyYWN0X3FrdihwaXhlbHMsIHNldHRpbmdzWyJsYXllciJdKQogICAgYiA9IHJlc3RvcmVkLmV4dHJhY3RfcWt2KHJlc3RvcmVkX3BpeGVscywgc2V0dGluZ3NbImxheWVyIl0pCiAgICBmb3Iga2V5IGluICgicSIsICJrIiwgInYiLCAicV9wYXRjaCIsICJrX3BhdGNoIiwgInZfcGF0Y2giKToKICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhW2tleV0sIGJba2V5XSwgcnRvbD0xZS01LCBhdG9sPTFlLTYpCiAgICBzY29yZXNfYSA9IGF0dGVudGlvbl9wYXRjaF9zY29yZXMoYVsicV9wYXRjaCJdWzBdLCBhWyJrIl1bMCwgMF0pCiAgICBzY29yZXNfYiA9IGF0dGVudGlvbl9wYXRjaF9zY29yZXMoYlsicV9wYXRjaCJdWzBdLCBiWyJrIl1bMCwgMF0pCiAgICBtYXNrX2EsIG1hc2tfYiA9IHNjb3Jlc19hID4gc2V0dGluZ3NbIlQxIl0sIHNjb3Jlc19iID4gc2V0dGluZ3NbIlQxIl0KICAgIGFzc2VydCB0b3JjaC5lcXVhbChtYXNrX2EsIG1hc2tfYikKICAgIGZhY2V0ID0gc2V0dGluZ3NbImZhY2V0Il0ubG93ZXIoKSArICJfcGF0Y2giCiAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhW2ZhY2V0XVswLCBtYXNrX2FdLCBiW2ZhY2V0XVswLCBtYXNrX2JdLCBydG9sPTFlLTUsIGF0b2w9MWUtNikKICAgIGNhbmRpZGF0ZSA9IGFbZmFjZXRdWzBdCiAgICBtYXRjaGVzX2EgPSBtdXR1YWxfbmVhcmVzdF9uZWlnaGJvcl9tYXRjaGVzKGFbZmFjZXRdWzAsIG1hc2tfYV0sIGNhbmRpZGF0ZSwgc2V0dGluZ3NbIlQyIl0pCiAgICBtYXRjaGVzX2IgPSBtdXR1YWxfbmVhcmVzdF9uZWlnaGJvcl9tYXRjaGVzKGJbZmFjZXRdWzAsIG1hc2tfYl0sIGNhbmRpZGF0ZSwgc2V0dGluZ3NbIlQyIl0pCiAgICBhc3NlcnQgdG9yY2guZXF1YWwobWF0Y2hlc19hLCBtYXRjaGVzX2IpCiAgICByZXBvcnQudXBkYXRlKGxvY2FsX3BhdGNoX3NoYXBlPWxpc3QoYltmYWNldF0uc2hhcGUpLCBzZWxlY3RlZF9sb2NhbF9jb3VudD1pbnQobWFza19iLnN1bSgpKSwKICAgICAgICAgICAgICAgICAgbG9jYWxfbWF4X2Fic29sdXRlX2RpZmZlcmVuY2U9ZmxvYXQoKGFbZmFjZXRdLWJbZmFjZXRdKS5hYnMoKS5tYXgoKSksIHN0cmljdF9sb2FkPVRydWUsCiAgICAgICAgICAgICAgICAgIG1ubl9tYXRjaGVzX2VxdWFsPVRydWUpCiAgICByZXR1cm4gcmVwb3J0CgoKZGVmIGV4cG9ydF9iZXN0X21vZGVsKG1vZGVsLCBjaGVja3BvaW50X3BhdGgsIG91dHB1dF9kaXIsIGNvbmZpZ3VyYXRpb24sIHZlcmlmaWNhdGlvbl9pbWFnZSk6CiAgICAiIiJMb2FkIHRoZSBmdWxsIGJlc3Qgc3RhdGUsIGV4cG9ydCBldmVyeSBwYXJhbWV0ZXIvYnVmZmVyLCBhbmQgdmVyaWZ5IHJlbG9hZC4iIiIKICAgIGNoZWNrcG9pbnQgPSB0b3JjaC5sb2FkKGNoZWNrcG9pbnRfcGF0aCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgIGlmICJtb2RlbF9zdGF0ZV9kaWN0IiBub3QgaW4gY2hlY2twb2ludDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBvcnQgcmVxdWlyZXMgYSBmdWxsIGJlc3QgbW9kZWxfc3RhdGVfZGljdDsgdHJhaW5hYmxlLW9ubHkgY2hlY2twb2ludHMgYXJlIG5vdCBzdXBwb3J0ZWQiKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNoZWNrcG9pbnRbIm1vZGVsX3N0YXRlX2RpY3QiXSwgc3RyaWN0PVRydWUpCiAgICBvdXRwdXRfZGlyID0gUGF0aChvdXRwdXRfZGlyKQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb25maWcgPSB7CiAgICAgICAgImZvcm1hdF92ZXJzaW9uIjogMSwgImFyY2hpdGVjdHVyZSI6ICJFZmZvVlBSIiwKICAgICAgICAibW9kZWwiOiB7ImJhY2tib25lX25hbWUiOiBtb2RlbC5iYWNrYm9uZS5tb2RlbF9uYW1lLCAiZ2xvYmFsX2RpbSI6IG1vZGVsLmdsb2JhbF9kaW0sCiAgICAgICAgICAgICJudW1fY2xhc3NlcyI6IG1vZGVsLm51bV9jbGFzc2VzLCAiZmluZV90dW5lX2xhc3Rfbl9sYXllcnMiOiBtb2RlbC5maW5lX3R1bmVfbGFzdF9uX2xheWVycywKICAgICAgICAgICAgIndpdGhfcmVnaXN0ZXJzIjogbW9kZWwuYmFja2JvbmUud2l0aF9yZWdpc3RlcnMsCiAgICAgICAgICAgICJjb3NmYWNlX3NjYWxlIjogbW9kZWwuY29zZmFjZS5zY2FsZSBpZiBtb2RlbC5jb3NmYWNlIGVsc2UgMzAuMCwKICAgICAgICAgICAgImNvc2ZhY2VfbWFyZ2luIjogbW9kZWwuY29zZmFjZS5tYXJnaW4gaWYgbW9kZWwuY29zZmFjZSBlbHNlIDAuNH0sCiAgICAgICAgImJhY2tib25lX2NvbmZpZyI6IG1vZGVsLmJhY2tib25lLm1vZGVsLmNvbmZpZy50b19kaWN0KCksCiAgICAgICAgInByb2Nlc3Nvcl9jb25maWciOiBtb2RlbC5iYWNrYm9uZS5pbWFnZV9wcm9jZXNzb3IudG9fZGljdCgpLAogICAgICAgICJoaWRkZW5fZGltIjogbW9kZWwuYmFja2JvbmUuaGlkZGVuX2RpbSwgInBhdGNoX3NpemUiOiBtb2RlbC5iYWNrYm9uZS5wYXRjaF9zaXplLAogICAgICAgICJpbmZlcmVuY2UiOiBjb25maWd1cmF0aW9uWyJpbmZlcmVuY2UiXSwgInJlcmFua2luZyI6IGNvbmZpZ3VyYXRpb25bInJlcmFua2luZyJdLAogICAgICAgICJjbGFzc19tYXBwaW5nIjogY2hlY2twb2ludC5nZXQoImNsYXNzX21hcHBpbmciKSwKICAgICAgICAic291cmNlIjogeyJjaGVja3BvaW50IjogUGF0aChjaGVja3BvaW50X3BhdGgpLm5hbWUsCiAgICAgICAgICAgICAgICAgICAic2VsZWN0aW9uX21ldHJpYyI6ICJtaW5pbXVtX21lYW5fdHJhaW5pbmdfbG9zcyIsCiAgICAgICAgICAgICAgICAgICAiYmVzdF90cmFpbmluZ19sb3NzIjogY2hlY2twb2ludC5nZXQoImJlc3RfdHJhaW5pbmdfbG9zcyIpfSwKICAgICAgICAidHJhbnNmb3JtZXJzX3ZlcnNpb24iOiB0cmFuc2Zvcm1lcnMuX192ZXJzaW9uX18sCiAgICB9CiAgICAjIGNsb25lIGFsc28gYnJlYWtzIHNoYXJlZCBzdG9yYWdlLCB3aGlsZSByZXRhaW5pbmcgZXZlcnkgc3RhdGVfZGljdCBlbnRyeS4KICAgIHN0YXRlID0ge25hbWU6IHRlbnNvci5kZXRhY2goKS5jcHUoKS5jb250aWd1b3VzKCkuY2xvbmUoKSBmb3IgbmFtZSwgdGVuc29yIGluIGNoZWNrcG9pbnRbIm1vZGVsX3N0YXRlX2RpY3QiXS5pdGVtcygpfQogICAgc2F2ZV9maWxlKHN0YXRlLCBzdHIob3V0cHV0X2RpciAvICJtb2RlbC5zYWZldGVuc29ycyIpLCBtZXRhZGF0YT17ImZvcm1hdCI6ICJwdCJ9KQogICAgZGVsIHN0YXRlLCBjaGVja3BvaW50CiAgICAob3V0cHV0X2RpciAvICJjb25maWcuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhjb25maWcsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJlcG9ydCA9IHZlcmlmeV9tb2RlbF9hcnRpZmFjdChtb2RlbCwgb3V0cHV0X2RpciwgdmVyaWZpY2F0aW9uX2ltYWdlKQogICAgKG91dHB1dF9kaXIgLyAiUkVBRE1FLm1kIikud3JpdGVfdGV4dCgKICAgICAgICAiIyBFZmZvVlBSXG5cbkZ1bGwgYmVzdC1jaGVja3BvaW50IHdlaWdodHMsIGluY2x1ZGluZyBmcm96ZW4gYmFja2JvbmUsIHByb2plY3Rpb24gYW5kIGNsYXNzaWZpZXIuXG4iCiAgICAgICAgIkFyY2hpdGVjdHVyZSBsaXZlcyBpbiB0aGUgcHJvamVjdCBQeXRob24gc291cmNlLiBObyBwcmV0cmFpbmVkIGRvd25sb2FkIGlzIG5lZWRlZC5cbiIKICAgICAgICBmIlVzZSB0cmFuc2Zvcm1lcnMge3RyYW5zZm9ybWVycy5fX3ZlcnNpb25fX30sIFB5VG9yY2ggYW5kIHNhZmV0ZW5zb3JzLlxuXG4iCiAgICAgICAgImBgYHB5dGhvblxuZnJvbSBzcmMuZWZmb3Zwci51dGlscy5tb2RlbF9hcnRpZmFjdCBpbXBvcnQgbG9hZF9tb2RlbF9hcnRpZmFjdCwgcHJlcGFyZV9pbWFnZVxuIgogICAgICAgICJpbXBvcnQgdG9yY2hcbm1vZGVsLCBjb25maWcgPSBsb2FkX21vZGVsX2FydGlmYWN0KCdhcnRpZmFjdHMvbW9kZWwnKVxuIgogICAgICAgICJwaXhlbHMgPSBwcmVwYXJlX2ltYWdlKG1vZGVsLCAnL3BhdGgvcXVlcnkuanBnJywgY29uZmlnWydpbmZlcmVuY2UnXVsncmVzb2x1dGlvbiddKVxuIgogICAgICAgICJ3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6XG4gICAgZW1iZWRkaW5nID0gbW9kZWwuZXh0cmFjdF9nbG9iYWwocGl4ZWxzKVxuIgogICAgICAgICIgICAgcWt2ID0gbW9kZWwuZXh0cmFjdF9xa3YocGl4ZWxzLCBjb25maWdbJ3JlcmFua2luZyddWydsYXllciddKVxuYGBgXG5cbiIKICAgICAgICAiVXNlX01vZGVsLmlweW5iIGFjY2VwdHMgcXVlcnkvZ2FsbGVyeSBmaWxlIHBhdGhzLCByZXRyaWV2ZXMgVG9wIDEwIHdpdGggZ2xvYmFsIGNvc2luZSBzaW1pbGFyaXR5LCAiCiAgICAgICAgInRoZW4gcmVyYW5rcyB0aG9zZSBjYW5kaWRhdGVzIGJ5IGxvY2FsIE1OTiBjb3VudCAodGllcyByZXRhaW4gZ2xvYmFsIHJhbmspLlxuIgogICAgICAgICJEb3dubG9hZCBjb25maWcuanNvbiBhbmQgbW9kZWwuc2FmZXRlbnNvcnMgYXMgc2VwYXJhdGUgZmlsZXM7IGRvIG5vdCB6aXAgdGhlIG1vZGVsLlxuXG4iCiAgICAgICAgIlZlcmlmaWNhdGlvbiBhZ2FpbnN0IGJlc3QgY2hlY2twb2ludDpcbmBgYGpzb25cbiIgKyBqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTIpICsgIlxuYGBgXG4iLAogICAgICAgIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludChqc29uLmR1bXBzKHsibW9kZWxfZGlyIjogc3RyKG91dHB1dF9kaXIpLCAic2FmZXRlbnNvcnNfZXhpc3RzIjogKG91dHB1dF9kaXIgLyAibW9kZWwuc2FmZXRlbnNvcnMiKS5pc19maWxlKCksCiAgICAgICAgICAgICAgICAgICAgICAidmVyaWZpY2F0aW9uIjogcmVwb3J0fSwgaW5kZW50PTIpKQogICAgcmV0dXJuIHJlcG9ydAo=', 'src/effovpr/reranking/mnn.py': 'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgdG9yY2gNCg0KDQpkZWYgbXV0dWFsX25lYXJlc3RfbmVpZ2hib3JfbWF0Y2hlcyhxdWVyeV9sb2NhbDogdG9yY2guVGVuc29yLCBjYW5kaWRhdGVfbG9jYWw6IHRvcmNoLlRlbnNvciwgdGhyZXNob2xkOiBmbG9hdCA9IDAuNjUpOg0KICAgIHF1ZXJ5X2xvY2FsID0gdG9yY2guYXNfdGVuc29yKHF1ZXJ5X2xvY2FsLCBkdHlwZT10b3JjaC5mbG9hdDMyKQ0KICAgIGNhbmRpZGF0ZV9sb2NhbCA9IHRvcmNoLmFzX3RlbnNvcihjYW5kaWRhdGVfbG9jYWwsIGR0eXBlPXRvcmNoLmZsb2F0MzIpDQogICAgaWYgcXVlcnlfbG9jYWwubmRpbSAhPSAyIG9yIGNhbmRpZGF0ZV9sb2NhbC5uZGltICE9IDI6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlF1ZXJ5IGFuZCBjYW5kaWRhdGUgbG9jYWwgZGVzY3JpcHRvcnMgbXVzdCBiZSAyRCB0ZW5zb3JzLiIpDQogICAgcV9ub3JtID0gcXVlcnlfbG9jYWwgLyAocXVlcnlfbG9jYWwubm9ybShkaW09MSwga2VlcGRpbT1UcnVlKSArIDFlLTgpDQogICAgY19ub3JtID0gY2FuZGlkYXRlX2xvY2FsIC8gKGNhbmRpZGF0ZV9sb2NhbC5ub3JtKGRpbT0xLCBrZWVwZGltPVRydWUpICsgMWUtOCkNCiAgICBzaW1pbGFyaXR5ID0gcV9ub3JtIEAgY19ub3JtLlQNCiAgICBpZiBzaW1pbGFyaXR5Lm51bWVsKCkgPT0gMDoNCiAgICAgICAgcmV0dXJuIHRvcmNoLmVtcHR5KCgwLCAyKSwgZHR5cGU9dG9yY2gubG9uZykNCiAgICBxdWVyeV90b19jYW5kaWRhdGUgPSBzaW1pbGFyaXR5LmFyZ21heChkaW09MSkNCiAgICBjYW5kaWRhdGVfdG9fcXVlcnkgPSBzaW1pbGFyaXR5LmFyZ21heChkaW09MCkNCiAgICBuX3F1ZXJpZXMgPSBxdWVyeV9sb2NhbC5zaXplKDApDQogICAgcXVlcnlfaWRzID0gdG9yY2guYXJhbmdlKG5fcXVlcmllcywgZGV2aWNlPXNpbWlsYXJpdHkuZGV2aWNlKQ0KICAgIG11dHVhbCA9IGNhbmRpZGF0ZV90b19xdWVyeVtxdWVyeV90b19jYW5kaWRhdGVdID09IHF1ZXJ5X2lkcw0KICAgIHZhbGlkX3Njb3JlcyA9IHNpbWlsYXJpdHlbcXVlcnlfaWRzLCBxdWVyeV90b19jYW5kaWRhdGVdW211dHVhbF0NCiAgICBrZWVwID0gdmFsaWRfc2NvcmVzID4gdGhyZXNob2xkDQogICAgcmV0YWluZWQgPSBxdWVyeV9pZHNbbXV0dWFsXVtrZWVwXQ0KICAgIG1hdGNoZWRfY2FuZGlkYXRlcyA9IHF1ZXJ5X3RvX2NhbmRpZGF0ZVttdXR1YWxdW2tlZXBdDQogICAgIyByZXR1cm4gYSBwYWlyIGxpc3QgKHF1ZXJ5X2luZGV4LCBjYW5kaWRhdGVfaW5kZXgpDQogICAgaWYgcmV0YWluZWQubnVtZWwoKSA9PSAwOg0KICAgICAgICByZXR1cm4gdG9yY2guZW1wdHkoKDAsIDIpLCBkdHlwZT10b3JjaC5sb25nLCBkZXZpY2U9c2ltaWxhcml0eS5kZXZpY2UpDQogICAgcmV0dXJuIHRvcmNoLnN0YWNrKFtyZXRhaW5lZCwgbWF0Y2hlZF9jYW5kaWRhdGVzXSwgZGltPTEpLmxvbmcoKQ0K'}
for relative_path, encoded_source in MODULE_FILES_B64.items():
    target = (Path("/kaggle/working/effovpr_use_model") / relative_path).resolve()
    if not target.is_relative_to(Path("/kaggle/working/effovpr_use_model").resolve()):
        raise ValueError(f"Refusing to write source outside /kaggle/working: {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(base64.b64decode(encoded_source))
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/hf_cache/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KAGGLE_INPUT = Path("/kaggle/input")
DATASET_ID = "PhamPhien/VN_Attractions"
DATASET_SPLIT = "train"
BATCH_SIZE = 1
TOP_K = 10
USE_LOCAL_MNN_RERANK = True
OUTPUT_DIR = Path("/kaggle/working/effovpr_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_dirs = []
for config_path in KAGGLE_INPUT.rglob("config.json"):
    candidate = config_path.parent
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if config.get("architecture") == "EffoVPR" and (candidate / "model.safetensors").is_file():
        model_dirs.append(candidate)
if len(model_dirs) != 1:
    raise ValueError(f"Expected exactly one EffoVPR artifact under /kaggle/input; found: {model_dirs}")
MODEL_DIR = model_dirs[0]
index_dirs = []
for metadata_path in KAGGLE_INPUT.rglob("gallery_metadata.parquet"):
    candidate = metadata_path.parent
    needed = ("gallery_features.npy", "index_config.json")
    if all((candidate / filename).is_file() for filename in needed):
        index_dirs.append(candidate)
if len(index_dirs) > 1:
    raise ValueError(f"Found multiple precomputed gallery bundles; attach only one: {index_dirs}")
PRECOMPUTED_DIR = index_dirs[0] if index_dirs else None
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
QUERY_PATHS = sorted(
    path for path in KAGGLE_INPUT.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    and MODEL_DIR not in path.parents
)
if not QUERY_PATHS:
    raise FileNotFoundError(f"No query images found under {KAGGLE_INPUT}; attach the Image Test 1 dataset as a Kaggle input.")
print("Model:", MODEL_DIR)
print("Precomputed gallery bundle:", PRECOMPUTED_DIR or "not found; embeddings will be computed")
print("Query images found:", len(QUERY_PATHS))
print("Code workspace:", CODE_ROOT)

## 3. Load the model and Hugging Face gallery

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
from IPython.display import display
from src.effovpr.utils.model_artifact import load_model_artifact
from src.effovpr.models.dino_attention import attention_patch_scores
from src.effovpr.reranking.mnn import mutual_nearest_neighbor_matches
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, artifact = load_model_artifact(MODEL_DIR, DEVICE)
RESOLUTION = int(artifact["inference"]["resolution"])
RERANK = artifact["reranking"]
gallery = load_dataset(DATASET_ID, split=DATASET_SPLIT)
required = {"id", "image", "label"}
missing = required.difference(gallery.column_names)
if missing:
    raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
print("Device:", DEVICE, "| resolution:", RESOLUTION)
print(f"Gallery rows: {len(gallery):,}")

## 4. Compute gallery embeddings and retrieve Top 10

In [ ]:
def preprocess(images):
    resized = [image.convert("RGB").resize((RESOLUTION, RESOLUTION), Image.Resampling.BICUBIC) for image in images]
    pixels = model.backbone.image_processor(images=resized, do_resize=False, do_center_crop=False, return_tensors="pt")["pixel_values"]
    return pixels.to(DEVICE)

@torch.inference_mode()
def encode(images):
    return model.extract_global(preprocess(images)).float()

if PRECOMPUTED_DIR is not None:
    index_config = json.loads((PRECOMPUTED_DIR / "index_config.json").read_text(encoding="utf-8"))
    gallery_metadata = pd.read_parquet(PRECOMPUTED_DIR / "gallery_metadata.parquet")
    gallery_features = np.load(PRECOMPUTED_DIR / "gallery_features.npy", allow_pickle=False).astype(np.float32)
    expected_dimension = int(artifact["model"]["global_dim"])
    checks = {
        "input_resolution": RESOLUTION,
        "feature_dimension": expected_dimension,
        "count": len(gallery_metadata),
        "normalized": True,
        "model_identifier": artifact["model"]["backbone_name"],
    }
    mismatches = {key: (index_config.get(key), value) for key, value in checks.items() if index_config.get(key) != value}
    if mismatches:
        raise ValueError(f"Precomputed gallery index does not match the model: {mismatches}")
    required_metadata = {"image_id", "label", "dataset_index"}
    if required_metadata.difference(gallery_metadata.columns):
        raise ValueError(f"Gallery metadata missing columns: {sorted(required_metadata.difference(gallery_metadata.columns))}")
    if gallery_features.ndim != 2 or gallery_features.shape != (len(gallery_metadata), expected_dimension):
        raise ValueError(f"Gallery features shape does not match metadata/model: {gallery_features.shape}")
    if int(index_config.get("count", -1)) != len(gallery_features):
        raise ValueError("Gallery index count does not match gallery_features.npy")
    if not np.isfinite(gallery_features).all():
        raise ValueError("Gallery features contain non-finite values")
    norms = np.linalg.norm(gallery_features, axis=1)
    if np.any(np.abs(norms - 1.0) > 1e-3):
        raise ValueError("Gallery features are not L2-normalized")

    # Validate row mapping using only ID/label columns; no image tensors are decoded here.
    gallery_info = gallery.select_columns(["id", "label"])
    source_ids = np.asarray(gallery_info["id"], dtype=str)
    source_labels = np.asarray(gallery_info["label"], dtype=str)
    gallery_dataset_indices = gallery_metadata["dataset_index"].to_numpy(dtype=np.int64)
    if np.any(gallery_dataset_indices < 0) or np.any(gallery_dataset_indices >= len(gallery_info)):
        raise ValueError("Gallery metadata dataset_index is outside the Hugging Face dataset")
    if not np.array_equal(source_ids[gallery_dataset_indices], gallery_metadata["image_id"].astype(str).to_numpy()):
        raise ValueError("Gallery IDs do not match the current Hugging Face dataset at dataset_index")
    if not np.array_equal(source_labels[gallery_dataset_indices], gallery_metadata["label"].astype(str).to_numpy()):
        raise ValueError("Gallery labels do not match the current Hugging Face dataset at dataset_index")
    gallery_ids = gallery_metadata["image_id"].astype(str).to_numpy()
    gallery_labels = gallery_metadata["label"].astype(str).to_numpy()
    print(f"Loaded precomputed {len(gallery_features):,}-image gallery vectors; skipped gallery model inference.")
    print("Ensure these features were built with this exact model checkpoint, not just the same architecture.")
else:
    # Fallback: compute features for the complete Hugging Face split.
    gallery_ids = np.asarray(gallery["id"], dtype=str)
    gallery_labels = np.asarray(gallery["label"], dtype=str)
    gallery_dataset_indices = np.arange(len(gallery), dtype=np.int64)
    feature_cache = Path("/kaggle/working/vn_attractions_gallery_embeddings.npz")
    cache_signature = f"{len(gallery)}:{RESOLUTION}:{(MODEL_DIR / 'model.safetensors').stat().st_size}"
    use_cache = False
    if feature_cache.is_file():
        saved = np.load(feature_cache, allow_pickle=False)
        if str(saved["signature"].item()) == cache_signature:
            gallery_ids = saved["ids"].astype(str)
            gallery_labels = saved["labels"].astype(str)
            gallery_features = saved["features"].astype(np.float32)
            use_cache = True
    if not use_cache:
        batches = []
        for start in range(0, len(gallery), BATCH_SIZE):
            rows = gallery[start:min(start + BATCH_SIZE, len(gallery))]
            batches.append(encode([im.convert("RGB") for im in rows["image"]]).cpu().numpy())
            if start == 0 or (start // BATCH_SIZE + 1) % 100 == 0 or start + BATCH_SIZE >= len(gallery):
                print(f"Gallery embeddings: {min(start + BATCH_SIZE, len(gallery)):,}/{len(gallery):,}")
        gallery_features = np.concatenate(batches).astype(np.float32)
        np.savez_compressed(feature_cache, signature=np.asarray(cache_signature), ids=gallery_ids, labels=gallery_labels, features=gallery_features)
    gallery_metadata = pd.DataFrame({"image_id": gallery_ids, "label": gallery_labels, "dataset_index": gallery_dataset_indices})

print("Gallery feature matrix:", gallery_features.shape, "| query images:", len(QUERY_PATHS))

## 5. Rank similar images and optionally rerank with MNN

In [ ]:
# Resolve this here too, so the retrieval cell does not depend on a prior alias assignment.
if "QUERY_PATHS" in globals():
    query_paths = list(QUERY_PATHS)
else:
    supported_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
    query_paths = sorted(
        path for path in Path("/kaggle/input").rglob("*")
        if path.is_file() and path.suffix.lower() in supported_extensions
        and MODEL_DIR not in path.parents
    )
if not query_paths:
    raise FileNotFoundError("No query images found in the mounted Kaggle inputs.")

@torch.inference_mode()
def local_descriptors(image):
    qkv = model.extract_qkv(preprocess([image]), layer=RERANK["layer"])
    scores = attention_patch_scores(qkv["q_patch"][0], qkv["k"][0, 0])
    selected = scores > float(RERANK["T1"])
    return qkv[f"{RERANK['facet'].lower()}_patch"][0, selected].float()
all_results = []
query_images = []
for query_path in query_paths:
    with Image.open(query_path) as opened:
        query_image = opened.convert("RGB")
    query_images.append((query_path, query_image))
    feature = encode([query_image])
    similarities = (gallery_features @ feature[0].cpu().numpy()).astype(np.float32)
    count = min(TOP_K, len(gallery))
    indices = np.argpartition(-similarities, count - 1)[:count]
    indices = indices[np.argsort(-similarities[indices])]
    q_local = local_descriptors(query_image) if USE_LOCAL_MNN_RERANK else None
    ranked = []
    for global_rank, idx in enumerate(indices, start=1):
        dataset_index = int(gallery_dataset_indices[idx])
        row = {"query_image": query_path.name, "image_id": str(gallery_ids[idx]), "label": str(gallery_labels[idx]),
               "cosine_similarity": float(similarities[idx]), "global_rank": global_rank,
               "mnn_matches": None, "gallery_index": int(idx), "dataset_index": dataset_index}
        if USE_LOCAL_MNN_RERANK:
            candidate = gallery[dataset_index]["image"].convert("RGB")
            local = local_descriptors(candidate)
            row["mnn_matches"] = int(mutual_nearest_neighbor_matches(q_local, local, threshold=float(RERANK["T2"])).shape[0])
        ranked.append(row)
    if USE_LOCAL_MNN_RERANK:
        ranked.sort(key=lambda row: (-row["mnn_matches"], row["global_rank"]))
    for rank, row in enumerate(ranked, start=1):
        row["final_rank"] = rank
    all_results.extend(ranked)
    print("Query:", query_path.name)
    display(pd.DataFrame([{k: row[k] for k in ("final_rank", "image_id", "label", "cosine_similarity", "mnn_matches")} for row in ranked]))

## 6. Display and save results

In [ ]:
for query_path, query_image in query_images:
    rows = [row for row in all_results if row["query_image"] == query_path.name]
    fig, axes = plt.subplots(2, 5, figsize=(18, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, rows):
        ax.imshow(gallery[int(row["dataset_index"])]["image"].convert("RGB"))
        title = f"#{row['final_rank']} {row['label']}\nID={row['image_id']}\ncos={row['cosine_similarity']:.3f}"
        if row["mnn_matches"] is not None:
            title += f" | MNN={row['mnn_matches']}"
        ax.set_title(title, fontsize=8)
    fig.suptitle(f"Query: {query_path.name}")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{query_path.stem}_top10.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)
export_rows = [{k: v for k, v in row.items() if k not in {"gallery_index", "dataset_index"}} for row in all_results]
results = pd.DataFrame(export_rows)
csv_path = OUTPUT_DIR / "top10_results.csv"
json_path = OUTPUT_DIR / "top10_results.json"
results.to_csv(csv_path, index=False, encoding="utf-8-sig")
json_path.write_text(results.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
print("CSV:", csv_path)
print("JSON:", json_path)
print("Image grids:", OUTPUT_DIR)